# **Preparing Dependancies**

In [ ]:
!pip install -q diffusers transformers accelerate safetensors
!pip install segment-anything

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
import gc # Penting untuk membersihkan RAM Colab
from PIL import Image, ImageDraw

# Import Modul dengan penataan yang lebih sistematis
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionImg2ImgPipeline,
    StableDiffusionInpaintPipeline,
    # Scheduler pilihan untuk efisiensi
    EulerAncestralDiscreteScheduler,
    DPMSolverMultistepScheduler,
    DDIMScheduler,
)

# Cek ketersediaan GPU (Sangat penting untuk debugging)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Fungsi helper untuk membersihkan memori (Gunakan ini setiap ganti fitur)
def flush_memory():
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
import sys
import importlib.metadata

# Dictionary to map pip package names to their module names/aliases in Python
# This helps in finding the correct version information.
package_map = [
    ("diffusers", "diffusers"),
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("safetensors", "safetensors"),
    ("segment-anything", "segment_anything"), # pip package name vs module name
    ("torch", "torch"),
    ("numpy", "np"), # numpy is commonly aliased as np
    ("matplotlib", "matplotlib"), # matplotlib.pyplot is often aliased as plt, but version is on main matplotlib module
    ("Pillow", "PIL"), # pip package name vs module name (PIL)
    ("opencv-python", "cv2"), # pip package name vs module name (cv2)
]

dependencies = {}

print("--- Hasil Pengecekan Versi Library ---")
for pip_name, module_alias_or_name in package_map:
    version = "Not Found"
    try:
        # Try getting version from importlib.metadata (most robust for installed packages)
        version = importlib.metadata.version(pip_name)
    except importlib.metadata.PackageNotFoundError:
        # If not found via pip name, try accessing __version__ from the module if imported
        module = None
        if module_alias_or_name in globals(): # Check for aliases like np, Image, cv2 if they are in global scope
            module = globals()[module_alias_or_name]
        elif module_alias_or_name in sys.modules: # Check for direct module imports
            module = sys.modules[module_alias_or_name]

        if module and hasattr(module, '__version__'):
            version = module.__version__
        elif pip_name == "matplotlib" and "matplotlib" in sys.modules and hasattr(sys.modules["matplotlib"], '__version__'):
            # Specific handling for matplotlib where __version__ is on the main module
            version = sys.modules["matplotlib"].__version__
    except Exception as e:
        version = f"Error: {e}"

    dependencies[pip_name] = version
    print(f"{pip_name.ljust(18)} : {version}")


# Membuat file requirements.txt secara otomatis
with open("requirements.txt", "w") as f:
    for lib, ver in dependencies.items():
        if ver != "Not Found" and not ver.startswith("Error"): # Only write if a valid version is found
            f.write(f"{lib}=={ver}\n")

print("\n[SUKSES] File 'requirements.txt' siap di-download!")

# **Kriteria 1: Melakukan Image Generation dari Teks (Text-to-Image)**

## **Load Base Pipeline Model**

In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"

# 1. Memuat pipeline utama (Text-to-Image) dengan optimasi penuh
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16, # Perbaikan parameter
    use_safetensors=True,
    safety_checker=None,
    low_cpu_mem_usage=True # Menghemat System RAM saat loading
)

# 2. Membuat pipeline INPAINT menggunakan komponen yang SAMA (Zero extra VRAM)
inpaint_pipe = StableDiffusionInpaintPipeline(
    vae=pipe.vae,
    text_encoder=pipe.text_encoder,
    tokenizer=pipe.tokenizer,
    unet=pipe.unet,
    scheduler=pipe.scheduler,
    feature_extractor=pipe.feature_extractor,
    safety_checker=None
)

# 3. Pindahkan ke GPU dan aktifkan fitur hemat memori
pipe.to("cuda")
inpaint_pipe.to("cuda") # Karena komponennya berbagi, ini tidak memakan memori ganda

# Optimasi tambahan untuk kartu grafis T4 di Colab
pipe.enable_attention_slicing()
inpaint_pipe.enable_attention_slicing()

# (Opsional) Jika masih mepet, aktifkan ini (tapi sedikit lebih lambat)
#pipe.enable_sequential_cpu_offload()

## **Generate Image**

In [ ]:
prompt = """
A lone astronaut standing on the dusty mars surface,
full white spacesuit with reflective black visor,
Earth rising large and green in the starry background sky,
cinematic lighting, ultra detailed, realistic texture,
wide angle composition, science fiction atmosphere,
high resolution, sharp focus, photorealistic masterpiece.
"""

main_negative_prompt = """
3d render, messy, blurry, low quality, bad art, ugly, sketch, grainy, unfinished, chromatic aberration"
"""



In [ ]:
def generate_simple_image(prompt, negative_prompt, seed):
    # Kriteria: Hanya Prompt, Negative Prompt, Seed
    generator = torch.Generator("cuda").manual_seed(seed)
    return pipe(
        prompt=prompt,
        negative_prompt=main_negative_prompt,
        generator=generator
    ).images[0]

def generate_advanced_image(prompt, negative_prompt, seed, guidance_scale, num_inference_step):
    # Kriteria: Tambahkan Guidance Scale & Steps
    generator = torch.Generator("cuda").manual_seed(seed)
    return pipe(
        prompt=prompt,
        negative_prompt=main_negative_prompt,
        generator=generator,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_step
    ).images[0]

Standard

In [ ]:
simple_img = generate_simple_image(
    prompt=prompt,
    negative_prompt=main_negative_prompt,
    seed=222
)

# Menampilkan hasil dengan ukuran yang lebih proporsional
plt.figure(figsize=(6, 6))
plt.imshow(simple_img)
plt.axis("off")
plt.show()

Advanced

In [ ]:
# Coba jalankan kembali
adv_img = generate_advanced_image(
    prompt=prompt,
    negative_prompt=main_negative_prompt,
    guidance_scale=7.5,
    num_inference_step=50,
    seed=222
)

plt.figure(figsize=(8, 8))
plt.imshow(adv_img)
plt.axis("off")
plt.show()

## **Generate Image with Hyperparameter Configuration**

## **Guidance Scale Comparison**

In [ ]:
guidance_val = [3, 7.5, 12]
images_guidance = []

# Tambahkan tqdm jika ingin melihat progress bar (opsional)
for scale in guidance_val:
    print(f"Generating image with scale: {scale}...")
    img_guide = generate_advanced_image(
        prompt=prompt,
        negative_prompt=main_negative_prompt,
        guidance_scale=scale,
        num_inference_step=50, # Saya turunkan ke 30 agar proses loop lebih cepat
        seed=222
    )
    images_guidance.append(img_guide)

# Visualisasi yang lebih rapi
plt.figure(figsize=(18, 6)) # Ukuran sedikit diperlebar agar tidak sesak
for i, img in enumerate(images_guidance):
    plt.subplot(1, 3, i + 1)
    plt.imshow(img)
    plt.title(f"Guidance Scale: {guidance_val[i]}", fontsize=14)
    plt.axis("off")

plt.tight_layout() # Memastikan antar subplot tidak tumpang tindih
plt.show()

### **Guidance Scale Explanation:**
1.Guidance Scale 3 (Low):

- Karakteristik: Gambar terlihat lebih "lembut" atau artistik, namun detail astronot mungkin kurang tegas. Earth di latar belakang mungkin tidak jelas atau se-jelas yang diminta prompt.

- Observasi: Model memiliki kebebasan kreatif tinggi, sehingga kepatuhan terhadap teks (prompt adherence) berada di level minimal.

2.Guidance Scale 7.5 (Balanced/Sweet Spot):

- Karakteristik: Ini adalah nilai standar industri. Keseimbangan antara detail baju astronot, refleksi pada visor, dan komposisi langit terjaga dengan baik dan komposisi bumi tampil.

- Observasi: Visual paling stabil, warna natural, dan semua elemen prompt (Astronaut, Mars, Earth) muncul secara proporsional.

3.Guidance Scale 12 (High):

- Karakteristik: Kontras warna akan meningkat tajam. Tekstur tanah Mars (regolith) terlihat sangat kasar dan tajam.

- Observasi: Gambar sangat patuh pada prompt, namun jika terlalu tinggi, bisa muncul artefak visual atau warna yang terlihat "terbakar" (oversaturated), visual dari bumi menjadi hilang.

## **Inference Steps Comparison**

In [ ]:
step_val = [10, 50]
images_step = []

for steps in step_val:
    print(f"Sedang memproses {steps} steps...") # Menambahkan log agar tahu proses berjalan
    img = generate_advanced_image(
        prompt=prompt,
        negative_prompt=main_negative_prompt,
        guidance_scale=7.5,
        num_inference_step=steps,
        seed=222
    )
    images_step.append(img)

# Visualisasi
plt.figure(figsize=(12, 6)) # 12x6 sudah cukup proporsional untuk 2 gambar
for i, img in enumerate(images_step):
    plt.subplot(1, 2, i + 1)
    plt.imshow(img)
    plt.title(f"Inference Steps: {step_val[i]}", fontsize=12)
    plt.axis("off")

plt.tight_layout() # Mencegah judul atau gambar terpotong
plt.show()

### **Inference Step Explanation:**

*   **Gambar dengan "Step" Rendah:**  
Gambar mungkin terlihat agak "smudgy" atau bertekstur cat air. Detail halus pada refleksi visor astronot atau tekstur regolith Mars belum terbentuk sempurna. Muncul sedikit noise karena proses dekoding dari latent ke pixel belum selesai sepenuhnya.


*   **Gambar dengan "Step" Tinggi:**  
Detail menjadi sangat tajam (sharp focus). Tekstur baju astronot terlihat lebih nyata, bayangan lebih halus, dan latar belakang bintang serta Bumi (Earth) tampak lebih jernih. Gambar mencapai stabilitas visual yang maksimal.

## **Batch Inference from One Prompt**

In [ ]:
def generate_batch_img(prompt, negative_prompt, num_inference_steps, guidance_scale, seed, num_img=4):
    # Perbaikan: Menggunakan argumen seed yang dilewatkan ke fungsi
    generator = torch.Generator("cuda").manual_seed(seed)

    # Gunakan inference_mode untuk efisiensi VRAM saat batching
    with torch.inference_mode():
        images_batch = pipe(
            prompt=prompt,
            negative_prompt=main_negative_prompt,
            generator=generator,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            num_images_per_prompt=num_img # Parameter penting untuk batching
        ).images

    return images_batch

def show_grid(images, title="Generated Images"):
    num_img = len(images)
    # Menghitung kolom dan baris secara otomatis (asumsi grid persegi atau mendekati)
    cols = 2
    rows = (num_img + cols - 1) // cols

    plt.figure(figsize=(10, 5 * rows)) # Tinggi menyesuaikan jumlah baris

    for i, img in enumerate(images):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Memberi ruang untuk suptitle
    plt.show()

In [ ]:
# Eksekusi
batch_results = generate_batch_img(
    prompt=prompt,
    negative_prompt=main_negative_prompt,
    num_inference_steps=50,
    guidance_scale=7.5,
    seed=222,
    num_img=2
)

show_grid(batch_results, title="Astronaut on Mars Variations")

## **Load Scheduler**

In [ ]:
def load_scheduler(pipe, scheduler_name):
    # Mengubah input menjadi lowercase untuk pengecekan yang lebih fleksibel
    name = scheduler_name.lower()

    if "euler" in name:
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
        selected = "EulerAncestral"

    elif "dpm" in name:
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
        selected = "DPM-Solver"

    elif "ddim" in name:
        pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
        selected = "DDIM"

    else:
        raise ValueError(f"Scheduler '{scheduler_name}' tidak valid. Pilih: Euler, DPM, atau DDIM.")

    print(f"✅ Scheduler berhasil diubah ke: {selected}")
    return pipe

###Euler A


In [ ]:
# 1. Ganti Scheduler
pipe = load_scheduler(pipe, "EulerAncestralDiscreteScheduler")

# 2. Generate Batch (Gunakan langkah yang lebih efisien jika ingin cepat)
images_euler = generate_batch_img(
    prompt=prompt,
    negative_prompt=main_negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    seed=222,
    num_img=2
)

# 3. Tampilkan Grid
show_grid(images_euler, "Comparison: Euler Ancestral Scheduler")

###DPMSolverMultistepScheduler

In [ ]:
# 1. Ganti ke DPM-Solver
pipe = load_scheduler(pipe, "DPMSolverMultistepScheduler")

# 2. Generate dengan langkah yang lebih efisien
# 25 steps sudah sangat cukup untuk DPM-Solver
image_dpm = generate_batch_img(
    prompt=prompt,
    negative_prompt=main_negative_prompt,
    num_inference_steps=25,
    guidance_scale=7.5,
    seed=222,
    num_img=2
)

# 3. Tampilkan hasil
show_grid(image_dpm, "DPM-Solver Multistep (Optimized)")

###DDIM

In [ ]:
# 1. Jalankan DDIM
pipe = load_scheduler(pipe, "DDIMScheduler")

image_ddim = generate_batch_img(
    prompt=prompt,
    negative_prompt=main_negative_prompt,
    num_inference_steps=50,
    guidance_scale=7.5,
    seed=222,
    num_img=2
)

show_grid(image_ddim, "DDIM Scheduler (Deterministic)")

# 2. PEMBERSIHAN MEMORI (WAJIB sebelum memuat SAM)
import gc
import torch

def flush_memory():
    gc.collect()
    torch.cuda.empty_cache()
    print("🧹 VRAM telah dibersihkan. Siap memuat model SAM!")

flush_memory()

### **Scheduler Comparation:**


1. Euler Ancestral (Euler A):

- Karakteristik: Menghasilkan gambar yang artistik dan halus. Karena bersifat stochastic (acak), scheduler ini memberikan variasi yang segar di setiap langkah.

- Observasi: Sangat efisien; pada 20-30 steps sudah bisa menghasilkan gambar berkualitas tinggi yang "lembut" di mata.

2. DPM++ (Multistep):

- Karakteristik: Dikenal sebagai scheduler tercepat dan paling detail saat ini. Hasilnya sangat stabil dan memiliki ketajaman tekstur yang unggul.

- Observasi: Merupakan pilihan terbaik untuk realisme. Dengan hanya 20 steps, hasilnya seringkali lebih baik daripada scheduler lain di 50 steps.

3. DDIM:

- Karakteristik: Bersifat deterministik dan konsisten. Pergerakan dari noise ke gambar sangat terukur dan kaku.

- Observasi: Sangat berguna untuk teknik tingkat lanjut seperti Inpainting atau Image-to-Image karena ia mampu menjaga struktur dasar gambar asli dengan sangat baik tanpa banyak perubahan acak.

# **Kriteria 2: Menyempurnakan Gambar Melalui Image-to-Image**

## **Base + Refiner Image Generation**

In [ ]:
# 1. Muat Base Model Sekali Saja
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "runwayml/stable-diffusion-v1-5"
dtype = torch.float16 if device == "cuda" else torch.float32

base_pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=dtype,
    use_safetensors=True
).to(device)

# 2. Buat Refiner Menggunakan Komponen dari Base_Pipe (Zero Extra VRAM)
# Kita memanggil constructor secara manual tanpa mendownload/memuat ulang model
refiner_pipe = StableDiffusionImg2ImgPipeline(
    vae=base_pipe.vae,
    text_encoder=base_pipe.text_encoder,
    tokenizer=base_pipe.tokenizer,
    unet=base_pipe.unet,
    scheduler=base_pipe.scheduler,
    feature_extractor=base_pipe.feature_extractor,
    safety_checker=base_pipe.safety_checker
).to(device)

# 3. Optimasi Tambahan
base_pipe.enable_attention_slicing()
refiner_pipe.enable_attention_slicing()

print("✅ Pipelines loaded with shared components. Memory saved!")

###Generate Base Image


In [ ]:
#Generate Base Image
generator = torch.Generator(device=device).manual_seed(222)

base_image = base_pipe(
    prompt=prompt,
    negative_prompt=main_negative_prompt,
    generator=generator,
).images[0]

base_image

In [ ]:
# Pastikan seed tetap sama agar AI "mengingat" struktur gambar awal
generator_refine = torch.Generator(device=device).manual_seed(222)

# Eksekusi Refiner (Image-to-Image)
refined_image = refiner_pipe(
    prompt=prompt,
    negative_prompt=main_negative_prompt,
    image=base_image,      # Menggunakan hasil Base Image tadi
    strength=0.35,         # 35% perubahan untuk menambah detail tekstur
    generator=generator_refine,
    num_inference_steps=50,
    guidance_scale=7.5
).images[0]

# Tampilkan Perbandingan
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1); plt.imshow(base_image); plt.title("Base Image (Original)"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(refined_image); plt.title("Refined Image (Enhanced)"); plt.axis("off")
plt.show()

## **Inpainting**

### **Load Model Inpainting**

In [ ]:
# 1. Tentukan device dan dtype
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# 2. Load Model (Tanpa memaksa safetensors)
print("⏳ Mengunduh/Memuat model Inpainting...")

pipe_inpaint = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=dtype,
    low_cpu_mem_usage=True
)

pipe_inpaint = pipe_inpaint.to(device)

# 3. Optimasi VRAM
pipe_inpaint.enable_attention_slicing()

print(f"✅ Model Inpainting Berhasil Dimuat di {device}!")

In [ ]:
# --- DEFINISI FUNGSI INPAINT_ENGINE (WAJIB BASIC) ---

#def inpaint_engine(pipe, prompt, image, mask):
#    """
#    Fungsi ini menerima input wajib: image, mask, dan prompt.
#    Menggunakan generator dengan Seed 9 sesuai instruksi.
#    """
#    # 1. Pastikan generator menggunakan Seed 9
#    generator = torch.Generator(device=device).manual_seed(9)
#
#    # 2. Pastikan Gambar & Masker dalam format yang benar (PIL & Resize)
#    # Stable Diffusion v1.5 bekerja optimal pada 512x512
#    image = image.convert("RGB").resize((512, 512))
#    mask = mask.convert("L").resize((512, 512))

#    # 3. Eksekusi Pipeline
#    print("✨ Memproses Inpainting...")
#    with torch.inference_mode():
#      result = pipe_inpaint(
#      prompt=prompt,
#      image=refined_image.convert("RGB"),      # Konversi ke RGB untuk stabilitas
#      mask_image=mask.convert("L"),    # Konversi ke Grayscale (L)
#    ).images[0]
#
#    return result

#print("✅ Fungsi inpaint_engine() didefinisikan dengan Seed 9.")

In [ ]:
# --- EKSEKUSI INPAINTING BASIC (2 pts) ---

# 1. Definisi Prompt (Wajib sesuai tema: Broken Satellite)
#satellite_prompt = "A detailed broken silver satellite, damaged metallic parts, wires, weathered texture, realistic shadows, matching mars environment"
#satellite_neg_prompt = "low quality, blurry, human, face, trees, water, text, watermark"

#mask = Image.new("L", (512, 512), 0)
#draw = ImageDraw.Draw(mask)
#mask_area = [240, 150, 480, 400] # Koordinat hardcode (Basic)
#draw.rectangle(mask_area, fill=255)


# 2. Panggil fungsi inpaint_engine yang sudah didefinisikan sebelumnya
# Pastikan menggunakan init_img_rfnd dan Seed 9
#final_basic_result = inpaint_engine(
#    pipe=pipe_inpaint,
#    prompt=satellite_prompt,
#    image=refined_image, # Gunakan hasil refinement
#    mask=mask
#)

# 3. Tampilkan Hasil
#plt.figure(figsize=(6, 6))
#plt.imshow(final_basic_result)
#plt.title("Basic Result: Inpainted Broken Satellite (Manual Mask)")
#plt.axis("off")
#plt.show()

### **Manual Masking**

In [ ]:
# 1. Pastikan ukuran konsisten
init_img_base = base_image.resize((512, 512))
init_img_rfnd = refined_image.resize((512, 512)) # Gunakan hasil refinement

# 2. Membuat Masker Manual
mask = Image.new("L", init_img_base.size, 0)
draw = ImageDraw.Draw(mask)

# Koordinat: [kiri, atas, kanan, bawah]
# Saya sesuaikan format koordinatnya agar standar PIL
mask_area = [240, 150, 480, 400]
draw.rectangle(mask_area, fill=255)

# 3. Visualisasi
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(init_img_base); plt.title("Base Image"); plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(init_img_rfnd); plt.title("Refined Image (Target)"); plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(mask, cmap="gray"); plt.title("Manual Mask Area"); plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def inpaint_engine(image, mask, prompt):
    # 1. Inisialisasi Generator dengan Seed Tetap
    generator = torch.Generator(device=device).manual_seed(9)

    # 2. Pastikan Gambar & Masker dalam format yang benar (PIL & Resize)
    # Stable Diffusion v1.5 bekerja optimal pada 512x512
    image = image.convert("RGB").resize((512, 512))
    mask = mask.convert("L").resize((512, 512))

    # 3. Proses Inpainting dengan Inference Mode (Hemat VRAM)
    print("✨ Memproses Inpainting...")
    with torch.inference_mode():
        result = pipe_inpaint(
            prompt=prompt,
            image=image,
            mask_image=mask,
            generator=generator,
            num_inference_steps=50,
            guidance_scale=7.5
        ).images[0]

    return result

In [ ]:
#PromptInpaint
prompt_inp = """
 highly detailed of a broken satellite with solar panel system,
 visible intricate internal wires, metallic debris, mechanical parts,
 realistic space junk, cinematic lighting, dramatic shadows, extreme detail, 8k, photorealistic
 """

In [ ]:
# --- EKSEKUSI ---
final_inpaint = inpaint_engine(
    image=init_img_rfnd,
    mask=mask,
    prompt=prompt_inp,
)

# Tampilkan Hasil Akhir
plt.figure(figsize=(8, 8))
plt.imshow(final_inpaint)
plt.title("Final Inpaint: Broken Satellite Added")
plt.axis("off")
plt.show()

### **Generate**

In [ ]:
#PromptInpaint
prompt_inp = """
 highly detailed of a broken satellite with solar panel system,
 visible intricate internal wires, metallic debris, mechanical parts,
 realistic space junk, cinematic lighting, dramatic shadows, extreme detail, 8k, photorealistic
 """

negative_prompt_inp = """
 smooth ground, empty moon surface, lunar soil, blurry,
 low resolution, bad anatomy, human, text, watermarks
 """

In [ ]:
def generate_inpaint_img(image, mask, prompt, negative_prompt, strength=1.0):
    # 1. Inisialisasi Generator dengan Seed Tetap
    generator = torch.Generator(device=device).manual_seed(9)

    # 2. Pastikan Gambar & Masker dalam format yang benar (PIL & Resize)
    # Stable Diffusion v1.5 bekerja optimal pada 512x512
    image = image.convert("RGB").resize((512, 512))
    mask = mask.convert("L").resize((512, 512))

    # 3. Proses Inpainting dengan Inference Mode (Hemat VRAM)
    print("✨ Memproses Inpainting...")
    with torch.inference_mode():
        result = pipe_inpaint(
            prompt=prompt,
            negative_prompt=negative_prompt,
            image=image,
            mask_image=mask,
            strength=strength, # Menentukan seberapa jauh AI mengubah area mask
            generator=generator,
            num_inference_steps=50,
            guidance_scale=7.5
        ).images[0]

    return result

In [ ]:
# --- EKSEKUSI ---
final_inpaint = generate_inpaint_img(
    image=init_img_rfnd,
    mask=mask,
    prompt=prompt_inp,
    negative_prompt=negative_prompt_inp,
    strength=1.0
)

# Tampilkan Hasil Akhir
plt.figure(figsize=(8, 8))
plt.imshow(final_inpaint)
plt.title("Final Inpaint: Broken Satellite Added")
plt.axis("off")
plt.show()

In [ ]:
# 1. Pastikan fungsi sudah didefinisikan (Sel sebelumnya)
# 2. Jalankan proses Inpainting untuk membuat variabel 'inpaint_refine'
inpaint_refine = generate_inpaint_img(
    image=init_img_rfnd,
    mask=mask,
    prompt=prompt_inp,
    negative_prompt=negative_prompt_inp,
    strength=1.0
)

# 3. Baru jalankan visualisasi (Sel yang error tadi)
plt.figure(figsize=(16, 8))
plt.subplot(1, 2, 1)
plt.imshow(init_img_rfnd)
plt.title("Before Inpaint")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(inpaint_refine)
plt.title("After Inpaint: Broken Satellite")
plt.axis("off")
plt.show()

## **Inpainting Menggunakan Automasking**

### **load Model Segmentation Untuk Masking**

In [ ]:
# Unduh model checkpoint resmi dari Meta AI
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O /content/sam_vit_b_01ec64.pth

import os
if os.path.exists("/content/sam_vit_b_01ec64.pth"):
    print("✅ File SAM Berhasil Diunduh!")
else:
    print("❌ Unduhan Gagal. Silakan cek koneksi internet Colab Anda.")

In [ ]:
import torch
import gc
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# 1. Pastikan VRAM benar-benar bersih sebelum memuat SAM
gc.collect()
torch.cuda.empty_cache()

# 2. Inisialisasi Model
sam_checkpoint = "/content/sam_vit_b_01ec64.pth"
model_type = "vit_b"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Memuat SAM {model_type} ke {device}...")
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
sam.eval() # Set ke mode evaluasi untuk menghemat memori

# 3. Inisialisasi Mask Generator
# Tips: points_per_side=32 adalah standar, kurangi ke 16 jika masih OOM
mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=32,
    pred_iou_thresh=0.86,
    stability_score_thresh=0.92,
    crop_n_layers=1,
    crop_n_points_downscale_factor=2,
    min_mask_region_area=100,  # Mengabaikan noise kecil
)

print("✅ SAM Mask Generator siap digunakan!")

In [ ]:
import gc
gc.collect()

torch.cuda.empty_cache()

#Perbaikan bagian SAM


### **Masking with Segmentation Model**

In [ ]:
from PIL import Image, ImageOps
import numpy as np

# 1. Konversi ke Numpy (Sudah benar)
img_np = np.array(inpaint_refine.convert("RGB"))

# 2. Generate masks (Sudah benar)
masks_auto = mask_generator.generate(img_np)

# 3. PERBAIKAN: Cari masker yang menutupi LANGIT
# Strategi: Langit biasanya ada di koordinat atas (misal [10, 10]).
# Kita akan mencari masker yang mengandung titik langit tersebut.
sky_mask_data = None
for m in masks_auto:
    # Cek apakah masker menutupi area pojok kiri atas (0,0)
    if m['segmentation'][10, 10] == True:
        sky_mask_data = m
        break

# Jika tidak ketemu dengan koordinat, gunakan masker terbesar lalu INVERT
if sky_mask_data is None:
    sky_mask_data = max(masks_auto, key=lambda x: x['area'])

# 4. Konversi ke Grayscale Mask
mask_uint8 = (sky_mask_data['segmentation'] * 255).astype(np.uint8)
auto_mask_obj = Image.fromarray(mask_uint8).convert("L")

# 5. PERBAIKAN KRUSIAL: Membalik Masker
# Sekarang area langit jadi PUTIH (untuk diganti satelit)
# area tanah jadi HITAM (untuk dijaga tetap pasir)
if auto_mask_obj.getpixel((10, 10)) == 0:
    auto_mask_final = ImageOps.invert(auto_mask_obj)
else:
    auto_mask_final = auto_mask_obj

# Tampilkan Perbandingan Baru
# visualisasi
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(inpaint_refine)
plt.title("Base Image")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(inpaint_refine)
plt.title("Refined Image")
plt.axis("off")

plt.figure(figsize=(10, 5))
plt.subplot(1, 3, 3)
plt.imshow(auto_mask_obj, cmap='gray')
plt.title("Original SAM (Detected Ground/Object)")

### **Generate**

In [ ]:
# --- Prompt Inpainting Baru (Fokus Langit/Galaksi) ---
prompt_inpaint_galaxy = """
a majestic, vibrant Milky Way galaxy core, detailed nebula structures,
millions of tiny stars, dramatic cosmic dust clouds, deep space photography,
realistic, cinematic lighting, purple and blue hues, breathtaking view, 8k, photorealistic
"""

# --- Negative Prompt Baru (Fokus Melindungi Astronot & Tanah) ---
# Area putih (langit) tidak boleh diisi oleh tanah atau merusak objek yang ada.
negative_prompt_inpaint_galaxy = """
moon surface, lunar soil, ground, dirt, rocks, smooth black sky, no stars,
blurry, low resolution, bad anatomy, human, text, watermarks,
distorted astronaut, clipping, cartoon
"""

In [ ]:
def inpaint_engine_galaxy(image, mask, prompt, negative_prompt, strength=0.9):

    # Gunakan seed yang sama untuk perbandingan yang konsisten
    generator = torch.Generator(device=device).manual_seed(9)

    result = pipe_inpaint(
        prompt=prompt,
        negative_prompt=negative_prompt, # Wajib dimasukkan
        image=image,
        mask_image=mask, # Gunakan mask langit yang sudah ada (putih di atas)
        num_inference_steps=50, # Naikkan sedikit step untuk detail bintang
        guidance_scale=30, # Naikkan agar AI lebih patuh pada prompt galaksi
        strength=1, # Gunakan 0.9 agar AI bebas membuat galaksi baru
        generator=generator
    ).images[0]

    return result

# --- Jalankan Inpainting Galaksi ---
# Gunakan auto_mask_full (mask yang putih di langit)
result_galaxy_base = inpaint_engine_galaxy(
    image=init_img_rfnd,
    mask=auto_mask_final,
    prompt=prompt_inpaint_galaxy,
    negative_prompt=negative_prompt_inpaint_galaxy,
    strength=0.9
)

result_galaxy_refined = inpaint_engine_galaxy(
    image=inpaint_refine,
    mask=auto_mask_final,
    prompt=prompt_inpaint_galaxy,
    negative_prompt=negative_prompt_inpaint_galaxy,
    strength=0.9
)

# Visualisasi Hasil
plt.figure(figsize=(12,6))

plt.subplot(1,2,1)
plt.imshow(result_galaxy_base)
plt.title("Galaxy Inpaint (Base)")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(result_galaxy_refined)
plt.title("Galaxy Inpaint (Refined)")
plt.axis("off")

## **Outpainting**

### **Prepare the Canvas**

In [ ]:
from PIL import Image, ImageDraw

def prepare_op_canvas(image, direction="bottom", expand_ratio=0.5):
    width, height = image.size

    # 1. Hitung penambahan pixel
    if direction in ["bottom", "top"]:
        expand_pixels = int(height * expand_ratio)
        new_width = width
        new_height = height + expand_pixels
    else: # right atau left
        expand_pixels = int(width * expand_ratio)
        new_width = width + expand_pixels
        new_height = height

    # 2. Pastikan dimensi kelipatan 8 (Wajib untuk Stable Diffusion)
    new_width = (new_width // 8) * 8
    new_height = (new_height // 8) * 8

    # 3. Buat Canvas Hitam
    canvas = Image.new("RGB", (new_width, new_height), (0, 0, 0))

    # 4. Buat Masker Hitam
    mask = Image.new("L", (new_width, new_height), 0)
    draw = ImageDraw.Draw(mask)

    # 5. Logika Penempelan & Masking berdasarkan Arah
    if direction == "bottom":
        canvas.paste(image, (0, 0))
        # Area putih adalah area baru di bawah
        draw.rectangle([(0, height), (new_width, new_height)], fill=255)

    elif direction == "right":
        canvas.paste(image, (0, 0))
        # Area putih adalah area baru di kanan
        draw.rectangle([(width, 0), (new_width, new_height)], fill=255)

    # Tambahkan arah lain (top/left) jika Anda ingin fitur "Zoom Out" penuh

    return canvas, mask

# --- EKSEKUSI ---
# Gunakan hasil Galaxy Inpaint Anda sebagai input
outpaint_input = result_galaxy_refined.resize((512, 512))

canvas_image, canvas_mask = prepare_op_canvas(
    outpaint_input, direction="bottom", expand_ratio=0.5
)

# Tampilkan Hasil Visualisasi
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(canvas_image)
plt.title("Canvas (Ready to Outpaint)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(canvas_mask, cmap="gray")
plt.title("Outpainting Mask (White = New Area)")
plt.axis("off")
plt.show()

### **Generate**

In [ ]:
# 1. Pastikan resolusi tetap terkontrol untuk menghindari OOM
# SD Inpainting bekerja paling baik jika salah satu sisi adalah 512
# Jika GPU Anda kuat, Anda bisa membiarkannya, tapi ini lebih aman:
prompt_op = """
detailed foreground of the Sun surface with various Fire scientific equipment,
cables lying on the dust, a deployable Sun seismometer, realistic Sun Lighting dan Fire ,
sharp details, cinematic lighting
"""

negative_prompt_op = """
sun, fire, yellow sky, bright yellow, smooth ground, empty ground,
blurry, distorted equipment, cartoon, missing details, bad lighting,
text, watermark, human, person
"""

# 2. Tambahkan Manual Seed agar hasil konsisten dengan instruksi (Seed 9)
generator = torch.Generator("cuda").manual_seed(9)

res_outpaint = pipe_inpaint(
    prompt=prompt_op,
    negative_prompt=negative_prompt_op,
    image=canvas_image,
    mask_image=canvas_mask,
    # Gunakan dimensi dari canvas_image yang sudah kita siapkan
    height=canvas_image.height,
    width=canvas_image.width,
    num_inference_steps=50,
    guidance_scale=8.5, # Naikkan sedikit agar prompt lebih ditaati
    generator=generator
).images[0]

# Tampilkan Hasil
plt.figure(figsize=(8,8)) # Sesuaikan aspect ratio tampilan
plt.imshow(res_outpaint)
plt.title("Advanced Outpainting Result (Mars Mission)")
plt.axis("off")
plt.show()

## **Outpainting Zoom Out**

### **Prepare Canvas for Zoom Out**

In [ ]:
from PIL import Image, ImageDraw

def prepare_op_canvas_zoom(image, direction, expand_ratio=0.5):
    width, height = image.size

    # Calculate new dimensions
    new_width, new_height = width, height
    offset_x, offset_y = 0, 0 # Where to paste the original image

    if direction == "bottom":
        new_height = int(height * (1 + expand_ratio))

        # Mask will cover the expanded bottom part
    elif direction == "top":
        new_height = int(height * (1 + expand_ratio))
        offset_y = new_height - height # Paste original image at the bottom of new canvas

        # Mask will cover the expanded top part
    elif direction == "right":
        new_width = int(width * (1 + expand_ratio))

        # Mask will cover the expanded right part
    elif direction == "left":
        new_width = int(width * (1 + expand_ratio))
        offset_x = new_width - width # Paste original image at the right of new canvas

    # Ensure dimensions are multiples of 8
    new_width = (new_width // 8) * 8
    new_height = (new_height // 8) * 8

    # Create black canvas and mask
    canvas = Image.new("RGB", (new_width, new_height), (0, 0, 0))
    mask = Image.new("L", (new_width, new_height), 0) # Black mask (0=no change)
    draw = ImageDraw.Draw(mask)

    # Paste original image onto canvas
    canvas.paste(image, (offset_x, offset_y))

    # Draw white rectangle on mask for the new area (255=change this area)
    if direction == "bottom":
        draw.rectangle([(0, height), (new_width, new_height)], fill=255)
    elif direction == "top":
        draw.rectangle([(0, 0), (new_width, offset_y)], fill=255)
    elif direction == "right":
        draw.rectangle([(width, 0), (new_width, new_height)], fill=255)
    elif direction == "left":
        draw.rectangle([(0, 0), (offset_x, new_height)], fill=255)

    return canvas, mask

print("✅ `prepare_op_canvas_zoom` function defined.")

### **Generate**

In [ ]:
import gc
import torch

# Gunakan salinan gambar terakhir Anda
zoom_img = res_outpaint.copy().resize((512, 512)) # Pastikan mulai dari 512
directions = ["bottom", "top", "left", "right"]

for d in directions:
    print(f"🚀 Processing direction: {d}...")

    # 1. Siapkan Canvas
    canvas, mask = prepare_op_canvas_zoom(
        zoom_img,
        direction=d,
        expand_ratio=0.4, # Ratio sedikit diperkecil agar tidak terlalu raksasa
    )

    # 2. Inpainting dengan Pengaturan Memori
    with torch.inference_mode():
        zoom_img = pipe_inpaint(
            prompt=prompt_op,
            negative_prompt=negative_prompt_op,
            image=canvas,
            mask_image=mask,
            # Paksa tinggi/lebar mengikuti canvas agar tidak error
            height=canvas.height,
            width=canvas.width,
            num_inference_steps=30, # Kurangi step sedikit untuk kecepatan
            guidance_scale=7.5,
            generator=torch.Generator("cuda").manual_seed(9)
        ).images[0]

    # 3. KUNCI ADVANCED: Resize kembali ke 512 agar iterasi selanjutnya aman
    zoom_img = zoom_img.resize((512, 512))

    # 4. Bersihkan VRAM
    gc.collect()
    torch.cuda.empty_cache()

# Tampilkan Hasil Akhir
plt.figure(figsize=(10, 10))
plt.imshow(zoom_img)
plt.title("Final Advanced Zoom Out (Optimized)")
plt.axis("off")
plt.show()